### WITH ROVER

In [1]:
from pynq.overlays.base import BaseOverlay

base = BaseOverlay("base.bit", download=True)

In [2]:
from pynq.lib.pmod import Pmod_IO
right_pins = [Pmod_IO(base.PMODA, i, 'out') for i in range(4)]
left_pins = [Pmod_IO(base.PMODB, i, 'out') for i in range(4)]

print("Motors successfully connected to PMOD headers")

Motors successfully connected to PMOD headers


In [3]:
%%microblaze base.ARDUINO
#include "i2c.h"

i2c mpu_device;
unsigned char mpu_addr = 0x68;

int init_mpu() {
    mpu_device = i2c_open_device(0); 
    
    unsigned char buf1[2] = {0x6B, 0x00};
    i2c_write(mpu_device, mpu_addr, buf1, 2);
    
    unsigned char buf2[2] = {0x1B, 0x00};
    i2c_write(mpu_device, mpu_addr, buf2, 2);
    
    unsigned char buf3[2] = {0x1A, 0x04};
    i2c_write(mpu_device, mpu_addr, buf3, 2);
    
    return 0;
}

int read_gyro_z() {
    unsigned char reg = 0x47; 
    unsigned char buf[2] = {0, 0};
    
    i2c_write(mpu_device, mpu_addr, &reg, 1);
    i2c_read(mpu_device, mpu_addr, buf, 2);
    
    short z_val = (buf[0] << 8) | buf[1];
    return (int)z_val;
}

In [4]:

import time
import math
step_sequence = [
    [1, 0, 0, 1],
    [0, 0, 1, 1],
    [0, 1, 1, 0],
    [1, 1, 0, 0],
]

_fwd = [step_sequence[i]     for i in range(4)]
_rev = [step_sequence[3 - i] for i in range(4)]

def release_motors():
    for i in range(4):
        left_pins[i].write(0)
        right_pins[i].write(0)

def _step_both(step_idx, l_fwd, r_fwd):
    s = step_idx % 4
    l = _fwd[s] if l_fwd else _rev[s]
    r = _fwd[s] if r_fwd else _rev[s]
    left_pins[0].write(l[0]);  right_pins[0].write(r[0])
    left_pins[1].write(l[1]);  right_pins[1].write(r[1])
    left_pins[2].write(l[2]);  right_pins[2].write(r[2])
    left_pins[3].write(l[3]);  right_pins[3].write(r[3])

class MPU6500_FPGA:
    def __init__(self):
        init_mpu()
        time.sleep(0.3)
        self.z_offset = 0.0

    def calibrate(self, samples=200):
        print("Calibrating IMU")
        total = 0.0
        count = 0
        for _ in range(samples):
            raw = read_gyro_z()
            if raw != 999999:
                total += raw
                count += 1
            time.sleep(0.01)
        self.z_offset = total / count if count > 0 else 0.0
        print(f"Calibration done. Offset: {self.z_offset:.3f} ({count}/{samples} good samples)")

    def get_dps(self):
        raw = read_gyro_z()
        if raw == 999999:
            return None
        return (raw - self.z_offset) / 131.0

imu = MPU6500_FPGA()
imu.calibrate()
print("Ready")


SQUARE_SIZE_CM       = 19.6
WHEEL_DIAM_CM        = 6.0
STEPS_PER_REV        = 200
STEPS_PER_CM         = STEPS_PER_REV / (math.pi * WHEEL_DIAM_CM)
STEPS_PER_SQUARE     = int(SQUARE_SIZE_CM * STEPS_PER_CM)

TURN_TARGET_DEG      = 88.3
DRIFT_THRESHOLD_DEG  = 1.5
MIN_SNAP_DEG         = 1.5 
TURN_ERROR_THRESHOLD = 0.5
ADJUSTMENT_MINUS     = 0.2

CORRECTION_DELAY     = 0.006
DRIVE_DELAY          = 0.002
TURN_DELAY           = 0.003


absolute_heading = 0.0

def _nearest_grid(heading):
    return round(heading / 90.0) * 90.0

def _update_heading(delta):
    global absolute_heading
    absolute_heading += delta

def _heading_error():
    return absolute_heading - _nearest_grid(absolute_heading)


def _integrate_gyro(angle, last_ts):
    now = time.time()
    dt  = now - last_ts
    dps = imu.get_dps()
    if dps is not None:
        angle += dps * dt
    return angle, now


def gyro_turn(target_deg, l_fwd, r_fwd, delay=TURN_DELAY, _is_correction=False):
    angle    = 0.0
    step_idx = 0
    last_ts  = time.time()
    deadline = time.time() + 12.0

    while abs(angle) < target_deg:
        if time.time() > deadline:
            print(f"  warning, turn timed out at {angle:.1f}° of {target_deg}°")
            break
        _step_both(step_idx, l_fwd, r_fwd)
        step_idx += 1
        angle, last_ts = _integrate_gyro(angle, last_ts)
        time.sleep(delay)

    release_motors()


    consecutive_still = 0
    coast_deadline = time.time() + 1.0
    while time.time() < coast_deadline:
        angle, last_ts = _integrate_gyro(angle, last_ts)
        dps = imu.get_dps()
        if dps is not None and abs(dps) < 2.0:
            consecutive_still += 1
        else:
            consecutive_still = 0
        if consecutive_still >= 3:
            break
        time.sleep(0.008)

    actual = abs(angle)
    print(f"  Turn complete: target={target_deg:.1f}° actual={actual:.1f}°")
    return actual

def _snap_to_grid():
    global absolute_heading
    error = _heading_error()
    if abs(error) < DRIFT_THRESHOLD_DEG:
        return


    correction = abs(error)
    if correction < MIN_SNAP_DEG:
        return

    print(f"  Grid snap: heading={absolute_heading:.1f}° error={error:+.1f}°. Correcting")

    if error > 0:
        actual = gyro_turn(correction, True, False,
                           delay=CORRECTION_DELAY, _is_correction=True)
        absolute_heading -= actual
    else:
        actual = gyro_turn(correction, False, True,
                           delay=CORRECTION_DELAY, _is_correction=True)
        absolute_heading += actual

    print(f"  Snapped: absolute={absolute_heading:.1f}°")

def gyro_turn_intentional(target_deg, l_fwd, r_fwd):
    actual = gyro_turn(target_deg, l_fwd, r_fwd, delay=TURN_DELAY)

    turn_error = actual - target_deg
    if abs(turn_error) > TURN_ERROR_THRESHOLD:
        direction = "overshot" if turn_error > 0 else "undershot"
        print(f"  → {direction} by {turn_error:+.1f}°. Micro-correcting…")
        correction_target = max(0, abs(turn_error) - ADJUSTMENT_MINUS)
        corr_actual = gyro_turn(correction_target,
                                not l_fwd if turn_error > 0 else l_fwd,
                                not r_fwd if turn_error > 0 else r_fwd,
                                delay=CORRECTION_DELAY,
                                _is_correction=True)
        if turn_error > 0:
            actual -= corr_actual
        else:
            actual += corr_actual

    signed_delta = -actual if (l_fwd and not r_fwd) else actual
    _update_heading(signed_delta)
    print(f"  absolute={absolute_heading:.1f}°")

    time.sleep(0.15)  
    _snap_to_grid()

def drive(steps, l_fwd=True, r_fwd=True, top_delay=DRIVE_DELAY):
    accel_steps = max(0, int(steps * 0.15))
    drift       = 0.0
    last_ts     = time.time()

    for s in range(steps):
        if s < accel_steps:
            frac  = s / accel_steps
            delay = 0.010 - (0.010 - top_delay) * frac
        elif s > steps - accel_steps:
            frac  = (s - (steps - accel_steps)) / max(accel_steps, 1)
            delay = top_delay + (0.010 - top_delay) * frac
        else:
            delay = top_delay
        _step_both(s, l_fwd, r_fwd)
        drift, last_ts = _integrate_gyro(drift, last_ts)
        time.sleep(delay)

    release_motors()
    time.sleep(0.30)

    _update_heading(drift)
    print(f"  Drive complete: drift={drift:+.1f}° absolute={absolute_heading:.1f}°")
    _snap_to_grid()


HEADING_MAP = {'UP': 0, 'RIGHT': 1, 'DOWN': 2, 'LEFT': 3}
current_heading = 0

def solve_maze(directions):
    global current_heading, absolute_heading

    absolute_heading = 0.0
    heading_names = {v: k for k, v in HEADING_MAP.items()}

    for raw_dir in directions:
        dir_str = raw_dir.strip().upper()
        if dir_str not in HEADING_MAP:
            print(f"Error: '{raw_dir}' is not a valid direction. Skipping.")
            continue

        target    = HEADING_MAP[dir_str]
        turn_diff = (target - current_heading) % 4

        if turn_diff == 1:
            print(f"Turning RIGHT toward {dir_str}…")
            gyro_turn_intentional(TURN_TARGET_DEG, True, False)
        elif turn_diff == 2:
            print(f"Turning 180° toward {dir_str}…")
            gyro_turn_intentional(TURN_TARGET_DEG * 2, True, False)
        elif turn_diff == 3:
            print(f"Turning LEFT toward {dir_str}…")
            gyro_turn_intentional(TURN_TARGET_DEG, False, True)

        current_heading = target
        print(f"Driving 1 square → now facing {heading_names[target]}…")
        drive(STEPS_PER_SQUARE)

    print(f"Maze complete. Final absolute heading: {absolute_heading:.1f}°")

release_motors()

Calibrating IMU
Calibration done. Offset: -49.405 (200/200 good samples)
Ready


In [5]:
current_heading = 0
# 0=Up, 1=Right, 2=Down, 3=Left

In [6]:
import requests
SERVER = 'http://13.48.68.209:5000'
print('Ready.')

Ready.


In [7]:
def move(direction):
    solve_maze([direction])
    
    resp = requests.post(f'{SERVER}/validate_move',
                         json={'direction': direction}, timeout=3)
    d = resp.json()
    new_pos = d.get('pos')
    
    requests.post(f'{SERVER}/log_rover',
                  json={'direction': direction, 'pos': new_pos}, timeout=3)

In [ ]:
while True:
    direction = input('> ').strip().upper()
    if direction == 'STOP':
        break
    if direction in ['UP', 'DOWN', 'LEFT', 'RIGHT']:
        move(direction)
    elif direction == 'CW':
        gyro_turn(1, True, False)
    elif direction == 'ACW':
        gyro_turn(1, False, True)

> RIGHT
> DOWN
> RIGHT
> DOWN
> RIGHT
> RIGHT
> DOWN
> LEFT
> UP


### WITHOUT ROVER

In [9]:
def move(direction):
    
    resp = requests.post(f'{SERVER}/validate_move',
                         json={'direction': direction}, timeout=3)
    d = resp.json()
    new_pos = d.get('pos')
    
    requests.post(f'{SERVER}/log_rover',
                  json={'direction': direction, 'pos': new_pos}, timeout=3)

In [10]:
while True:
    direction = input('> ').strip().upper()
    if direction == 'STOP':
        break
    if direction in ['UP', 'DOWN', 'LEFT', 'RIGHT']:
        move(direction)
    elif direction == 'CW':
        gyro_turn(1, True, False)
    elif direction == 'ACW':
        gyro_turn(1, False, True)

> UP
> RIGHT
> RIGHT
> RIGHT
> UP
> UP
> UP
> UP
> LEFT
> RIGHT
> UP
> RIGHT
> RIGHT
> DOWN
> LEFT
> UP
> RIGHT
> DOWN
> UP
> RIGHT


KeyboardInterrupt: Interrupted by user